# Typhoon ASR Realtime: local CUDA inference

This notebook transcribes local audio with Typhoon ASR. It keeps only the safe placeholder `sample_audio.mp3`; provide the actual file at runtime through the `ASR_SAMPLE_AUDIO` environment variable or replace the placeholder locally.

The public model downloads and caches automatically on the first run. Do not hard-code tokens, passwords, or absolute local paths in this notebook.

## Installation

Install the package once in the notebook kernel environment:

```bash
pip install -U typhoon-asr
```

In [1]:
import contextlib
import io
import os
from pathlib import Path

import torch
from typhoon_asr import transcribe

AUDIO_PATH = Path(os.environ.get("ASR_SAMPLE_AUDIO", "sample_audio.mp3"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(
        "Audio file not found. Set ASR_SAMPLE_AUDIO or place sample_audio.mp3 beside this notebook."
    )

print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
# Suppress package diagnostics so the published notebook stores only the useful result.
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    result = transcribe(str(AUDIO_PATH), device=DEVICE)

print("Transcript:")
print(result["text"])
print(f"\nAudio duration: {result['audio_duration']:.2f} s")
print(f"ASR processing time: {result['processing_time']:.2f} s")

Transcript:
ศูนย์ต่อต้านการฉ้อโกงออนไลน์นะคะส่งข้อมูลให้ตํารวจตามจับกุมวัยรุ่นค่ะที่มีการถอนเงินหน้าธนาคารเพื่อส่งไปให้บัญชีม้านะคะ ขณะที่ภาพรวมตลอดทั้งสัปดาห์มีคนถูกหลอกโอนเงินรวมความเสียหายกว่าสามร้อยเก้าสิบเจ็ดล้านบาท นี่เป็นหนึ่งในคดีที่ศูนย์ต่อต้านการฉ้อโกงออนไลน์ประสานตํารวจในพื้นที่ สภ เชียงดาว จังหวัดเชียงใหม่ จับกุมวัยรุ่นชายอายุสิบเจ็ดและสิบเก้าปี หลังรับงานกดเงินสดถอนเงินให้กับแก๊งสแกมเมอร์จากธนาคารแห่งหนึ่ง เบื้องต้นพบของกลางเงินสด หนึ่งแสนห้าหมื่นบาท สมุดบัญชีธนาคาร บัตร ATM และโทรศัพท์มือถือสองเครื่อง สอบถามผู้ต้องหารับว่าได้รับงานกด ถอนเงินให้กับชายวัยรุ่นอีกคนอายุสิบเก้าปี ซึ่งก่อนหน้านี้ก็เพิ่งไปถอนเงินสดหนึ่งแสนบาทจากพื้นที่ สภ.นาหวายให้ไป เลิกกับค่าตอบแทนห้าพันบาท ขณะที่ภาพรวมมีการจับกุมผู้ต้องหารวมสิบเอ็ดคดียึดของกลางเงินสดได้รวมกว่าห้าล้านสี่แสนบาท ขณะที่ภาพรวมวันที่ห้าถึงสิบเอ็ดเมษายนที่ผ่านมา พบว่ามีผู้เสียหายแจ้งความผ่านระบบออนไลน์รวมกว่าเจ็ดพันสามร้อยคดี ความเสียหายรวมกว่าสามร้อยเก้าสิบเจ็ดล้านบาท ในจํานวนนี้เป็นคดีหลอกลงทุนสร้างความเสียหายมากที่สุดรวมกว่าสองร้

## Notes

- `processing_time` reported by the package measures transcription after the model and audio preparation steps; it is not the full cold-start time.
- Preserve the raw transcript before applying numeric normalization.
- See the [Typhoon ASR model card](https://huggingface.co/typhoon-ai/typhoon-asr-realtime) for current usage and requirements.